In [2]:
import tkinter as tk
from tkinter import scrolledtext, messagebox, filedialog, PhotoImage
import json
import threading
from datetime import datetime
import os
import random 

# --- IMPORT CAMERA/OPENCV COMPONENTS ---
# NOTE: You must have 'opencv-python' and 'numpy' installed.
import cv2
import numpy as np

# -------------------------------------
# CONFIGURATION & CONSTANTS
# -------------------------------------

ALERT_FILE = "hq_alerts.json" 

# Base64 data for the embedded logo (a blue "Star of Life" icon)
LOGO_IMAGE_BASE64 = """
R0lGODlhgACAAOMPAAAAABgYGCAgICAgIDAwMDIyMjQ0NDY2Nj4+Pj4+PkBAQEJCQkJCQlJSUlRU
VFVVVVtbW11dXV5eXl9fX2BgYGFhYWJiYmNjY2RkZGVlZWdnZ2lpaWpqamxsbG5ubm9vb3Jy
c3R0dHZ2dnd3d3h4eHl5eXp6enx8fH5+foCAgIGBgYKCgoODg4SEhIWFhYqKioyMjI2NjY6O
jpOTk5SUlJWVlZaWlpeXl5iYmJmZmZqampubm5ycnJ2dnZ6enp+fn6CgoKKioqOjo6SkpKampqen
p6ioqKmpqaqqqqurq6ysrK2tra6urq+vr7CwsLGxsbKysrS0tLW1tba2tre3t7i4uLm5ubu7
u7y8vL29vb6+vr+/v8DAwMTExMnJydDQ0NfX19jY2NnZ2dra2tvb29zc3N3d3d7e3uLi4ubm
5u/v7/Pz8/b29vj4+Pr6+v7+/v///wAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACH5
BAEKAA8ALAAAAACAAIAAAwT/MNlJq7046827/wNf91KCAYAs2YFsmZpmqpnOurds3/Ns6H5u
C/N87w/6Y7Gi+KxBHBSSyeJymUkplVq9YcMicVlMLpprduut1GYwNltO6/Y7HC6b0/M8PmfR
6fU+v5/v+/+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6yt
rq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxsfIycrLzM3Oz9DR0tPU1deX2Nna29zd3t/g4eLj
5OXm5+jp6uvs7e7v8PHy8/T19vf4+fr7/P3+/wADih1ItKDBgwgTKlzIsKHDhxAjSpxIsaLF
ixgzaowVAAAABGzcuJFCgACEDgBAjpw44kAAQ5EeCIBi5UlEBAQAbADCw8mPAQEAUPCp5UGK
AiUeMJgCBAAUBgAcHAgxANWGEBA6VFDQYkKFAhMqNIjBAtavWFEeIEDAQAe2WAYeDCAgQMAI
bLkS6NBBwYICbRsAIEBgwd8KDBIWQBChwYQDBwIeChw4sOHDhyAfFnBAucADCAYwHMBA+uiP
AwswLJjBwUCJBAcYRBCwIIGEzggYfAhA4ECFzhE4uBxBw4MCAgYnQBxB4kKJzyU8rFCgQ4IC
DBIqXDDwwUCDCRQqXPhw4UKDBxc6fPiw4UKIESVUuFCgQoUEESZUuHChQYMGDRg0aNCgQYMG
CRgwUMCAgAACEQBACH5BAEKAA8ALAAAAACAAIAAAwT/MNlJq7046827/wNf91KCAYAs2YFsmZpm
qpnOurds3/Ns6H5uC/N87w/6Y7Gi+KxBHBSSyeJymUkplVq9YcMicVlMLpprduut1GYwNltO
6/Y7HC6b0/M8PmfR6fU+v5/v+/+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6Ch
oqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxsfIycrLzM3Oz9DR0tPU1deX
2Nna29zd3t/g4eLj5OXm5+jp6uvs7e7v8PHy8/T19vf4+fr7/P3+/wADih1ItKDBgwgTKlwo
EAAEBwQQHAgxIEGBCRMmNFjA4ECIEBUeGChRosGBBg0mJghA4ECJByUeMJhC5UeCBAw4mHBB
gYCKByUeMJhyZYEBBAQAbADyQ8mPAQEAUPCp5UGKAiUeMJgCBAAUBgAcHAgxANWGEBA6VFDQ
YkKFAhMqNIjBAtavWFEeIEDAQAe2WAYeDCAgQMAIbLkS6NBBwYICbRsAIEBgwd8KDBIWQBCh
wYQDBwIeChw4sOHDhyAfFnBAucADCAYwHMBA+uiPAwswLJjBwUCJBAcYRBCwIIGEzggYfAhA
4ECFzhE4uBxBw4MCAgYnQBxB4kKJzyU8rFCgQ4ICDBIqXDDwwUCDCRQqXPhw4UKDBxc6fPiw
4UKIESVUuFCgQoUEESZUuHChQYMGDRg0aNCgQYMGCRgwUMCAgAACEQBACH5BAEKAA8ALAAAAACA
AIAAAwT/MNlJq7046827/wNf91KCAYAs2YFsmZpmqpnOurds3/Ns6H5uC/N87w/6Y7Gi+KxBHBSS
yeJymUkplVq9YcMicVlMLpprduut1GYwNltO6/Y7HC6b0/M8PmfR6fU+v5/v+/+AgYKDhIWG
h4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8
vb6/wMHCw8TFxsfIycrLzM3Oz9DR0tPU1deX2Nna29zd3t/g4eLj5OXm5+jp6uvs7e7v8PHy
8/T19vf4+fr7/P3+/wADih1ItKDBgwgTKlw4EAKCBwQQHAgxIEGBCRMmNFjA4ECIEBUeGChR
osGBBg0mJghA4ECJByUeMJhC5UeCBAw4mHBBgYCKByUeMJhyZYEBBAQAbADyQ8mPAQEAUPCp
5UGKAiUeMJgCBAAUBgAcHAgxANWGEBA6VFDQYkKFAhMqNIjBAtavWFEeIEDAQAe2WAYeDCAg
QMAIbLkS6NBBwYICbRsAIEBgwd8KDBIWQBChwYQDBwIeChw4sOHDhyAfFnBAucADCAYwHMBA
+uiPAwswLJjBwUCJBAcYRBCwIIGEzggYfAhA4ECFzhE4uBxBw4MCAgYnQBxB4kKJzyU8rFCg
Q4ICDBIqXDDwwUCDCRQqXPhw4UKDBxc6fPiw4UKIESVUuFCgQoUEESZUuHChQYMGDRg0aNCg
QYMGCRgwUMCAgAACEQBACH5BAEKAA8ALAAAAACAAIAAAwT/MNlJq7046827/wNf91KCAYAs2YFs
mZpmqpnOurds3/Ns6H5uC/N87w/6Y7Gi+KxBHBSSyeJymUkplVq9YcMicVlMLpprduut1GYw
NltO6/Y7HC6b0/M8PmfR6fU+v5/v+/+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2e
n6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxsfIycrLzM3Oz9DR0tPU
1deX2Nna29zd3t/g4eLj5OXm5+jp6uvs7e7v8PHy8/T19vf4+fr7/P3+/wADih1ItKDBgwgT
Klw4sIEBCAUeMJjwgMCFEA0iSJxIsaLFihctPNDiwIgHAgBAjBw5IoEAgwgoR5o8IoEBAgIC
ABgAseFEAgICAHgYseHCAgMCKgwAseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgIC
AHiYseHCAgICAHgYseHCAgICAHgYsdGBAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgIC
AHiYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYsdGBAgICAHgYseHCAgICAHgYseHCAgIC
AHiYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgIC
AHiYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYsdGBAgIC
AHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgIC
AHiYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYseHCAgICAHgYsdGNBgwYMGjRo0KBBgwcM
GDhgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwcM
GDhgwIABAgYMGDBg0KBBgwYMGDBgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYM
GDBg0KBBgwYMGjRo0KBBgwcMGDBgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYM
GDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYM
GDBg0KBBgwYMGjRo0KBBgwcMGDBgwIABAgYMGDBg0KBBgwYMGjRo0KBBgwYMGDBgwIABAgYM
GS0BAAA7
"""

# --- CORE LOGIC FUNCTIONS (Based on PPT Requirements) ---
# (These functions are global and are called by the Console class)

def triage_vitals_to_severity(vitals):
    """
    Simulates the Feedforward Neural Network (FNN) or Random Forest classification.
    (PPT Requirement: Automated Triage Classification)
    """
    try:
        hr = int(vitals.get("HR", 0))
        spo2 = int(vitals.get("SpO2", 0))
        
        # Critical Thresholds (PPT: SpO2 < 85%, HR > 120 bpm)
        if spo2 < 90 and hr > 120:
            return "Critical", 0.95 
        elif spo2 < 95 or hr > 100:
            return "Warning", 0.70
        else:
            return "Normal", 0.99
    except ValueError:
        return "Invalid Data", 0.0

def get_first_aid_from_model(severity):
    """
    Simulates the Multi-Output Neural Network/Decision Tree Logic.
    (PPT Requirement: Real-Time, Action-Oriented Medical Guidance)
    """
    if severity == "Critical":
        return ["APPLY TOURNIQUET IMMEDIATELY (Hemorrhage Protocol)", "Alert HQ/Evacuation Team", "Maintain Airway and Breathing"]
    elif severity == "Warning":
        return ["Stabilize wound and control minor bleeding", "Monitor vitals every 5 minutes", "Administer pain relief if possible"]
    elif severity == "Normal":
        return ["Basic wound care", "Continue monitoring", "Await transport"]
    else:
        return ["Data Error: Manual review required."]

def write_alert(severity, message, first_aid_actions, patient_id="Unknown", bbox=None, app_instance=None):
    """
    Logs alert to file and sends it to the main GUI thread for display.
    (PPT Requirement: Seamless Communication & Geospatial Tracking)
    """
    alert = {
        "patient_id": patient_id,
        "alert_id": f"alert-{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}",
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "severity": severity,
        "message": message,
        "first_aid": first_aid_actions,
        "bbox": bbox,
        "gps_coords": "LAT: 33.123, LON: -117.456" # Placeholder for Geospatial Tracking
    }
    # Append to local log file
    with open(ALERT_FILE, "r+") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            data = []
        data.append(alert)
        f.seek(0)
        json.dump(data, f, indent=2)
    
    # Send alert message to GUI
    if app_instance and severity in ["Critical", "Warning"]:
        alert_message = f"!!! {severity.upper()} ALERT !!! {message} (Patient: {patient_id})"
        app_instance.log_to_system_panel(alert_message, tag="ALERT")

# --- CORE OPENCV/VISION LOGIC (From Code 2) ---

def red_mask_from_frame(frame):
    """Detects red areas in the frame (simulated injury) using HSV color range."""
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    lower1 = np.array([0, 60, 40])
    upper1 = np.array([8, 255, 255])
    lower2 = np.array([170, 60, 40])
    upper2 = np.array([180, 255, 255])
    mask1 = cv2.inRange(hsv, lower1, upper1)
    mask2 = cv2.inRange(hsv, lower2, upper2)
    mask = cv2.bitwise_or(mask1, mask2)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7,7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask

def get_bboxes_from_mask(mask, min_area=600):
    """Extracts bounding boxes for detected injury contours."""
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area >= min_area:
            x, y, w, h = cv2.boundingRect(cnt)
            bboxes.append({"x": int(x), "y": int(y), "w": int(w), "h": int(h), "area": int(area)})
    return bboxes

# Global list to simulate data from multiple (wearable) sources
SIMULATED_PATIENT_DATA = {
    "SOLDIER-001": {"HR": 80, "SpO2": 98, "Last_Update": datetime.now()},
    "SOLDIER-002": {"HR": 105, "SpO2": 95, "Last_Update": datetime.now()},
    "SOLDIER-003": {"HR": 135, "SpO2": 88, "Last_Update": datetime.now()},
}

def simulate_vitals_update(patient_id, is_critical_area, is_warning_area):
    """
    Simulates dynamic vitals updates for a patient.
    (PPT Requirement: Continuous Monitoring)
    """
    data = SIMULATED_PATIENT_DATA.get(patient_id, {"HR": 80, "SpO2": 98})

    if is_critical_area:
        # High-Risk vitals
        data["HR"] = random.randint(130, 150)
        data["SpO2"] = random.randint(80, 89)
    elif is_warning_area:
        # Moderate-Risk vitals
        data["HR"] = random.randint(100, 120)
        data["SpO2"] = random.randint(90, 95)
    else:
        # Normal fluctuation
        data["HR"] = random.randint(75, 90)
        data["SpO2"] = random.randint(97, 100)
    
    data["Last_Update"] = datetime.now()
    SIMULATED_PATIENT_DATA[patient_id] = data
    return data

def process_casualty(frame, patient_id, app_instance):
    """
    Core processing loop.
    This simulates visual detection on the *live frame* and integrates with
    *simulated wearable vitals* to determine triage.
    """
    mask = red_mask_from_frame(frame)
    bboxes = get_bboxes_from_mask(mask)
    
    # Determine risk from visual input
    is_critical_area = bboxes and bboxes[0]['area'] > 15000
    is_warning_area = bboxes and 6000 < bboxes[0]['area'] <= 15000

    # Simulate Vitals update for the patient currently in view
    vitals = simulate_vitals_update(patient_id, is_critical_area, is_warning_area)

    # Triage Decision based on the combined simulated Vitals
    severity, _ = triage_vitals_to_severity(vitals)
    first_aid = get_first_aid_from_model(severity)
    
    message = "Massive bleeding suspected." if is_critical_area else "Injury detected." if is_warning_area else "Vitals OK."

    if severity in ["Critical", "Warning"]:
        write_alert(severity, message, first_aid, patient_id, bboxes, app_instance)

    return severity, bboxes, first_aid, vitals

def get_triage_for_all_patients(app_instance):
    """
    Simulates running the triage model for ALL known patients (Multiple Soldier Analysis).
    The one with the worst status is prioritized for the GUI dashboard.
    """
    triage_results = []
    
    # In a real app, this would iterate over live data streams (not a static list)
    for patient_id, vitals in SIMULATED_PATIENT_DATA.items():
        severity, confidence = triage_vitals_to_severity(vitals)
        first_aid = get_first_aid_from_model(severity)
        triage_results.append({
            "id": patient_id,
            "vitals": vitals,
            "severity": severity,
            "confidence": confidence,
            "first_aid": first_aid
        })
        
        # Log critical alerts for non-visual patients (e.g., detected by wearable only)
        if severity == "Critical" and (datetime.now() - vitals["Last_Update"]).total_seconds() < 5:
             write_alert(severity, f"Vitals report CRITICAL. HR: {vitals['HR']}, SpO2: {vitals['SpO2']}", first_aid, patient_id, app_instance=app_instance)

    # Simple Prioritization: Find the worst status
    priorities = {"Critical": 3, "Warning": 2, "Normal": 1, "Invalid Data": 0}
    prioritized_patient = max(triage_results, key=lambda x: priorities.get(x['severity'], 0))
    
    return triage_results, prioritized_patient

def run_triage_camera(app_instance):
    """The main loop for the OpenCV Triage System, runs in a separate window."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        app_instance.log_to_system_panel("CRITICAL ERROR: Could not open camera. Check connection.", tag="ERROR")
        return

    font = cv2.FONT_HERSHEY_SIMPLEX
    app_instance.log_to_system_panel("Triage Camera activated in separate window. Press 'q' to stop.", tag="SYSTEM")

    # Cycle through simulated patient IDs for the live feed
    patient_ids = list(SIMULATED_PATIENT_DATA.keys())
    current_patient_id = patient_ids[0]
    patient_index = 0
    update_count = 0

    while True:
        ret, frame = cap.read()
        if not ret: 
            app_instance.log_to_system_panel("WARNING: Video frame dropped.", tag="SYSTEM")
            break

        # Cycle the patient in the visual feed every 100 frames to simulate multi-casualty switching
        if update_count > 0 and update_count % 100 == 0:
            patient_index = (patient_index + 1) % len(patient_ids)
            current_patient_id = patient_ids[patient_index]
            app_instance.log_to_system_panel(f"Live feed switched to: {current_patient_id}", tag="SYSTEM")
        
        # 1. Process the live visual frame for the current patient
        severity, bboxes, first_aid_list, live_vitals = process_casualty(frame, current_patient_id, app_instance)
        
        # 2. Re-Triage ALL patients (Multi-Casualty Analysis)
        all_results, prioritized_patient = get_triage_for_all_patients(app_instance)
        
        # 3. Update the main GUI (Console) with the HIGHEST priority patient data
        if prioritized_patient['id'] != current_patient_id and update_count % 100 == 0:
            app_instance.log_to_system_panel(f"Priority Shift: Displaying status for {prioritized_patient['id']} (Severity: {prioritized_patient['severity']})", tag="PRIORITY")

        # Update dashboard only on change to avoid GUI lag
        if update_count % 10 == 0: # Update GUI every 10 frames
            app_instance.update_guidance_dashboard(
                prioritized_patient['id'], 
                prioritized_patient['severity'], 
                prioritized_patient['vitals'], 
                prioritized_patient['first_aid'],
                all_results
            )

        # 4. OpenCV display logic (Unchanged)
        overlay = frame.copy()
        color_code = (0, 255, 0) 
        if severity == "Critical": color_code = (0, 0, 255) 
        elif severity == "Warning": color_code = (0, 165, 255) 

        if severity != "Normal":
            for bb in bboxes:
                x, y, w, h = bb["x"], bb["y"], bb["w"], bb["h"]
                cv2.rectangle(overlay, (x, y), (x + w, y + h), color_code, 2)
        
        header = f"LIVE FEED: {current_patient_id} | TRIAGE: {severity} | HR: {live_vitals['HR']}"
        cv2.putText(overlay, header, (10, 30), font, 0.8, color_code, 2, cv2.LINE_AA)
        
        cv2.imshow("Field Medic Triage View", overlay)
        
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        
        update_count += 1
        
    cap.release()
    cv2.destroyAllWindows()
    app_instance.log_to_system_panel("Triage Camera shut down.", tag="SYSTEM")
    app_instance.update_guidance_dashboard(
        "Camera Feed Ended", 
        "SYSTEM", 
        {"HR": "---", "SpO2": "---"}, 
        ["Awaiting next launch..."],
        []
    )


# -------------------------------------
# TKINTER WELCOME PAGE CLASS
# -------------------------------------

class WelcomePage(tk.Frame):
    """The first page (splash screen) of the application."""
    def __init__(self, master, start_callback):
        super().__init__(master, bg="#2c2c2c") # Dark background
        self.start_callback = start_callback
        
        # Load and store the logo image
        self.logo_image_tk = PhotoImage(data=LOGO_IMAGE_BASE64)

        # Logo Display
        tk.Label(
            self, 
            image=self.logo_image_tk, 
            bg="#2c2c2c"
        ).pack(pady=(50, 20))
        
        # Main title
        tk.Label(
            self, 
            text="AI-ENABLED MILITARY MEDICAL", 
            font=("Helvetica", 32, "bold"), 
            bg="#2c2c2c", 
            fg="#e0e0e0"
        ).pack(pady=(10, 5))
        
        tk.Label(
            self, 
            text="PRIORITIZATION SYSTEM", 
            font=("Helvetica", 32, "bold"), 
            bg="#2c2c2c", 
            fg="#556b2f" # Use the Army Green as accent
        ).pack(pady=(0, 30))
        
        tk.Label(
            self, 
            text="Combat Medical Prioritization Hub", 
            font=("Helvetica", 16), 
            bg="#2c2c2c", 
            fg="#a0a0a0"
        ).pack(pady=(0, 60))

        # Start Button
        start_button = tk.Button(
            self, 
            text="START SYSTEM", 
            command=self.start_callback,
            font=("Helvetica", 14, "bold"),
            bg="#e74c3c", # Red button
            fg="white",
            relief=tk.FLAT,
            padx=20,
            pady=10
        )
        start_button.pack()
        
        tk.Label(
            self, 
            text="Ensure all camera and sensor hardware is connected.", 
            font=("Helvetica", 11), 
            bg="#2c2c2c", 
            fg="#707070"
        ).pack(side=tk.BOTTOM, pady=20)


# -------------------------------------
# TKINTER CONSOLE CLASS (The Main GUI)
# -------------------------------------

class MilitaryTriageConsole(tk.Frame):
    def __init__(self, master):
        super().__init__(master) # Initialize the Frame
        self.master = master # Store master for root-level ops (like 'after')
        
        # --- THEME DEFINITION ---
        self.theme_palettes = {
            'light': {
                'root_bg': '#f4f7fb',
                'panel_bg': '#ffffff',
                'header_bg': '#2b6ea3',
                'header_fg': 'white',
                'text_fg': '#000000',
                'log_bg': '#1e1e1e',
                'log_fg': '#00ff00',
                'button_bg': '#d0d0d0',
                'button_fg': '#000000',
                'tag_critical': '#e74c3c',
                'tag_warning': '#f39c12',
                'tag_normal_bg': '#27ae60', # Background for dashboard
                'tag_normal_fg': '#27ae60', # Foreground for list
                'tag_system': '#2b6ea3'
            },
            'dark': {
                'root_bg': '#2c2c2c',
                'panel_bg': '#3a3a3a',
                'header_bg': '#2b6ea3', 
                'header_fg': 'white',
                'text_fg': '#e0e0e0',
                'log_bg': '#1e1e1e', 
                'log_fg': '#00ff00', 
                'button_bg': '#505050',
                'button_fg': '#e0e0e0',
                'tag_critical': '#ff6060', 
                'tag_warning': '#f39c12', 
                'tag_normal_bg': '#30db70',
                'tag_normal_fg': '#30db70',
                'tag_system': '#4b9cc3'  
            },
            'army_green': {
                'root_bg': '#3d3d3d', # Dark neutral grey
                'panel_bg': '#4a4a4a', # Lighter neutral grey
                'header_bg': '#556b2f', # Dark Olive Green
                'header_fg': '#ffffff',
                'text_fg': '#e0e0e0',  # Off-white/light beige
                'log_bg': '#1e1e1e',   # Black
                'log_fg': '#9acd32',   # Yellow-green
                'button_bg': '#556b2f', # Dark Olive Green
                'button_fg': '#ffffff',
                'tag_critical': '#e74c3c', # Bright Red
                'tag_warning': '#f39c12', # Orange
                'tag_normal_bg': '#556b2f', # Olive
                'tag_normal_fg': '#9acd32', # Yellow-green
                'tag_system': '#87ceeb'  # Sky Blue
            }
        }
        self.current_theme = 'army_green' # Default theme
        # ---------------------------

        self.setup_ui()
        self.set_status("System Initialized. Ready to launch camera.")
        
        # Initial call to populate the dashboard with system status
        self.update_guidance_dashboard(
            "SYSTEM", 
            "INITIALIZING", 
            {"HR": "N/A", "SpO2": "N/A"}, 
            ["Launch the Triage Camera (🔴) to begin live monitoring."],
            []
        )
    
    def setup_ui(self):
        # All widgets are created here and stored as self attributes
        # Note: Widgets are packed into 'self' (the frame) not 'self.master' (the root)
        
        self.main_monitor_frame = tk.Frame(self)
        self.main_monitor_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)

        self.top_container = tk.Frame(self.main_monitor_frame)
        self.top_container.pack(fill=tk.BOTH, expand=True)

        # --- LEFT (GUIDANCE DASHBOARD) ---
        self.guidance_frame = tk.Frame(self.top_container, bd=1, relief=tk.RIDGE)
        self.guidance_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0, 10))

        self.header2 = tk.Label(self.guidance_frame, text="HIGHEST PRIORITY MEDICAL GUIDANCE", font=("Helvetica", 14, "bold"))
        self.header2.pack(fill=tk.X)

        self.content_container = tk.Frame(self.guidance_frame)
        self.content_container.pack(fill=tk.BOTH, expand=True, padx=8, pady=8)

        self.guidance_box = scrolledtext.ScrolledText(self.content_container, wrap=tk.WORD, font=("Helvetica", 11), state=tk.DISABLED, height=15)
        self.guidance_box.pack(fill=tk.BOTH, expand=True)

        # --- RIGHT (MULTI-PATIENT LIST) ---
        self.multi_patient_frame = tk.Frame(self.top_container, bd=1, relief=tk.RIDGE, width=350)
        self.multi_patient_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=False)
        self.multi_patient_frame.pack_propagate(False) 

        self.header3 = tk.Label(self.multi_patient_frame, text="MULTI-CASUALTY STATUS", font=("Helvetica", 14, "bold"))
        self.header3.pack(fill=tk.X)

        self.patient_list_box = scrolledtext.ScrolledText(self.multi_patient_frame, wrap=tk.WORD, font=("Courier New", 10), state=tk.DISABLED, padx=5, pady=5)
        self.patient_list_box.pack(fill=tk.BOTH, expand=True, padx=5, pady=5)

        # --- FOOTER SECTION (Status + Button) ---
        self.footer_frame = tk.Frame(self)
        self.footer_frame.pack(fill=tk.X, padx=10, pady=(5, 0))

        self.status_label = tk.Label(self.footer_frame, text="", anchor="w")
        self.status_label.pack(side=tk.LEFT, fill=tk.X, expand=True)

        # Theme Toggle Button
        self.toggle_theme_button = tk.Button(self.footer_frame, text="Toggle Theme", command=self.toggle_theme, font=("Helvetica", 10))
        self.toggle_theme_button.pack(side=tk.LEFT, padx=10, pady=4)

        # Launch Button
        self.launch_camera_button = tk.Button(self.footer_frame, text="🔴 Launch Triage Camera", command=self.launch_camera_async, bg="#e74c3c", fg="white", font=("Helvetica", 10, "bold"))
        self.launch_camera_button.pack(side=tk.RIGHT, padx=10, pady=4)

        # --- BOTTOM PANEL: System & Alert Log ---
        self.run_label = tk.Label(self, text="System & Alert Log (HQ Communication)", font=("Helvetica", 13, "bold"))
        self.run_label.pack(fill=tk.X, padx=10, pady=(0,2))

        self.alert_log = scrolledtext.ScrolledText(self, wrap=tk.WORD, font=("Courier New", 11), height=8)
        self.alert_log.pack(fill=tk.X, expand=False, padx=10, pady=(0,10))
        self.alert_log.config(state=tk.DISABLED)

        # --- APPLY THEME ---
        self.apply_theme() # Apply the default theme

    # --- THEME MANAGEMENT ---
    
    def toggle_theme(self):
        """Switches the application theme between light, dark, and army_green."""
        if self.current_theme == 'light':
            self.current_theme = 'dark'
        elif self.current_theme == 'dark':
            self.current_theme = 'army_green'
        else: # Must be army_green
            self.current_theme = 'light'
            
        self.apply_theme()
        self.log_to_system_panel(f"Theme switched to {self.current_theme.upper()}", tag="SYSTEM")

    def apply_theme(self):
        """Applies all colors from the currently selected theme palette."""
        palette = self.theme_palettes[self.current_theme]
        
        # Apply to root and frames
        self.master.configure(bg=palette['root_bg']) # Configure the root window
        self.configure(bg=palette['root_bg']) # Configure self (the main frame)
        self.main_monitor_frame.configure(bg=palette['root_bg'])
        self.top_container.configure(bg=palette['root_bg'])
        self.footer_frame.configure(bg=palette['root_bg'])
        
        # Panel Backgrounds
        self.guidance_frame.configure(bg=palette['panel_bg'])
        self.content_container.configure(bg=palette['panel_bg'])
        self.multi_patient_frame.configure(bg=palette['panel_bg'])
        
        # Headers
        self.header2.configure(bg=palette['header_bg'], fg=palette['header_fg'])
        self.header3.configure(bg=palette['header_bg'], fg=palette['header_fg'])
        self.run_label.configure(bg=palette['header_bg'], fg=palette['header_fg'])
        
        # Status/Toggle
        self.status_label.configure(bg=palette['root_bg'], fg=palette['text_fg'])
        self.toggle_theme_button.configure(bg=palette['button_bg'], fg=palette['button_fg'])

        # --- Guidance Box ---
        self.guidance_box.configure(state=tk.NORMAL)
        self.guidance_box.configure(bg=palette['panel_bg'], fg=palette['text_fg'], 
                                    insertbackground=palette['text_fg']) # Add cursor color
        # Configure tags
        self.guidance_box.tag_config("status_tag_Critical", foreground='white', background=palette['tag_critical'], font=("Helvetica", 13, "bold"))
        self.guidance_box.tag_config("status_tag_Warning", foreground='white', background=palette['tag_warning'], font=("Helvetica", 13, "bold"))
        self.guidance_box.tag_config("status_tag_Normal", foreground='white', background=palette['tag_normal_bg'], font=("Helvetica", 13, "bold"))
        self.guidance_box.tag_config("status_tag_INITIALIZING", foreground='white', background=palette['tag_system'], font=("Helvetica", 13, "bold"))
        self.guidance_box.tag_config("status_tag_SYSTEM", foreground='white', background=palette['tag_system'], font=("Helvetica", 13, "bold"))
        
        self.guidance_box.tag_config("header_tag", font=("Helvetica", 11, "underline"))
        self.guidance_box.tag_config("patient_tag", font=("Helvetica", 12, "underline"))
        self.guidance_box.configure(state=tk.DISABLED)

        # --- Patient List Box ---
        self.patient_list_box.configure(state=tk.NORMAL)
        self.patient_list_box.configure(bg=palette['panel_bg'], fg=palette['text_fg'],
                                        insertbackground=palette['text_fg'])
        # Configure tags
        self.patient_list_box.tag_config("list_header", font=("Courier New", 10, "bold"))
        self.patient_list_box.tag_config("Critical", foreground=palette['tag_critical'])
        self.patient_list_box.tag_config("Warning", foreground=palette['tag_warning'])
        self.patient_list_box.tag_config("Normal", foreground=palette['tag_normal_fg'])
        self.patient_list_box.configure(state=tk.DISABLED)

        # --- Alert Log ---
        self.alert_log.configure(state=tk.NORMAL)
        # FIX: The repeated keyword argument 'fg' is removed here.
        self.alert_log.configure(bg=palette['log_bg'], fg=palette['log_fg']) 
        self.alert_log.configure(state=tk.DISABLED)


    # --- New Functionality: Update Dashboard ---
    
    def update_guidance_dashboard(self, patient_id, severity, vitals, first_aid, all_results):
        """Updates the main guidance panel and the multi-patient list."""
        
        # 1. Update HIGHEST PRIORITY GUIDANCE BOX
        self.guidance_box.config(state=tk.NORMAL)
        self.guidance_box.delete("1.0", tk.END)

        # Status text and tag are now dynamic
        status_text = f"Status: {severity.upper()}"
        status_tag_name = f"status_tag_{severity}"
        
        if severity == "Critical":
            status_text = f"!!! CRITICAL EMERGENCY - IMMEDIATE ACTION REQUIRED !!!"
        elif severity == "Warning":
            status_text = f"WARNING - CLOSE MONITORING REQUIRED"
        
        # Display Triage Status and Vitals (PPT: Detailed Vitals Display)
        self.guidance_box.insert(tk.END, f"PRIORITY PATIENT: {patient_id}\n", "patient_tag")
        self.guidance_box.insert(tk.END, f"{status_text}\n", status_tag_name)
        self.guidance_box.insert(tk.END, f"Vitals: HR={vitals.get('HR')}, SpO2={vitals.get('SpO2')}\n\n")

        # Display Actions
        self.guidance_box.insert(tk.END, "--- RECOMMENDED ACTIONS ---\n", "header_tag")
        for i, action in enumerate(first_aid):
            self.guidance_box.insert(tk.END, f"• {action}\n")
            
        self.guidance_box.config(state=tk.DISABLED)

        # 2. Update MULTI-PATIENT LIST (PPT: Multiple Soldier Analysis)
        self.patient_list_box.config(state=tk.NORMAL)
        self.patient_list_box.delete("1.0", tk.END)
        
        self.patient_list_box.insert(tk.END, f"--- {len(all_results)} CASUALTY MONITOR ---\n", "list_header")
        
        # Sort results to show highest severity first
        priorities = {"Critical": 3, "Warning": 2, "Normal": 1, "Invalid Data": 0}
        sorted_results = sorted(all_results, key=lambda x: priorities.get(x['severity'], 0), reverse=True)

        for result in sorted_results:
            pid = result['id']
            sev = result['severity']
            hr = result['vitals'].get('HR', 'N/A')
            spo2 = result['vitals'].get('SpO2', 'N/A')
            
            line = f"\n{pid} ({sev.upper()})\n HR:{hr:<4} SpO2:{spo2}\n"
            self.patient_list_box.insert(tk.END, line, sev) # Use severity as tag name

        self.patient_list_box.config(state=tk.DISABLED)

    # --- Utility Methods ---
        
    def launch_camera_async(self):
        """Starts the OpenCV camera loop in a separate thread."""
        for t in threading.enumerate():
            if t.name == "TriageCameraThread" and t.is_alive():
                 messagebox.showinfo("Camera Running", "The Triage Camera is already active in a separate window.")
                 return

        t = threading.Thread(target=run_triage_camera, args=(self,), daemon=True, name="TriageCameraThread")
        t.start()
        self.log_to_system_panel("Attempting to initialize camera feed...", tag="SYSTEM")
        self.set_status("TTriage Camera running (check separate window)")
        
    def log_to_system_panel(self, message, tag="INFO"):
        """Thread-safe method to log any message to the main GUI's alert log."""
        log_message = f"[{datetime.now().strftime('%H:%M:%S')}] [{tag}] {message}\n"
        self.master.after(0, self._insert_to_log_panel, log_message) # Use self.master.after

    def _insert_to_log_panel(self, log_message):
        """Private method that performs the actual GUI update."""
        self.alert_log.config(state=tk.NORMAL)
        self.alert_log.insert(tk.END, log_message)
        self.alert_log.see(tk.END)
        self.alert_log.config(state=tk.DISABLED)

    def set_status(self, txt):
        self.status_label.config(text=f"Status: {txt}")

# -------------------------------------
# MAIN APP CONTROLLER
# -------------------------------------

class AppController:
    def __init__(self, root):
        self.root = root
        self.root.title("AI Triage System")
        self.root.geometry("1100x750")
        self.root.configure(bg="#2c2c2c") # Set initial bg for splash
        
        # Start with the welcome page
        self.show_welcome_page()

    def show_welcome_page(self):
        self.welcome_frame = WelcomePage(self.root, self.start_application)
        self.welcome_frame.pack(fill=tk.BOTH, expand=True)

    def start_application(self):
        # 1. Destroy the welcome frame
        self.welcome_frame.destroy()
        
        # 2. Create and pack the main application
        # The MilitaryTriageConsole is now a Frame itself, so it's packed *into* the root.
        self.main_app_frame = MilitaryTriageConsole(self.root)
        self.main_app_frame.pack(fill=tk.BOTH, expand=True)
        
        # Set the title for the main app
        self.root.title("AI Military Triage Console - Combat Medical Prioritization Hub")


# --- Main Application Execution ---

if __name__ == "__main__":
    if not os.path.exists(ALERT_FILE):
        with open(ALERT_FILE, "w") as f:
            json.dump([], f, indent=2)
            \
            
            
    root = tk.Tk()
    app = AppController(root)
    root.mainloop()